# ISMN stations → NISAR GCOV **full frames** → EASE2 **50 m sigma0** (MAAP DPS)

This is a **new batch-controller notebook**, adapted from `Indonesia_NISAR_MAAP_DPS_Batch_FullScenes(1).ipynb`, that targets the **316 point coordinates** in `ismn_gt50_coords_points_june.csv` (not Indonesia). The `n_points_june_onwards` field is carried as metadata; it is **not** a NISAR acquisition count.

## What the notebook does

1. Loads the station CSV (`network, station, lat, lon, n_points_june_onwards`).
2. Searches **only** the official ASF NISAR GCOV collections `NISAR_L2_GCOV_PROVISIONAL_V1` and `NISAR_L2_GCOV_V1` from **2026-06-01 through the execution date**. No BETA or urgent-response collections.
3. Matches each station to CMR granule footprints; records the matched stations for **each whole frame**, including overlapping A/D acquisitions. Searches use spatial bins **only for discovery**.
4. Deduplicates repeated search results and superseded processing versions of the same acquisition; prefers **validated** over **provisional** when the same acquisition appears in both.
5. Produces a **granule manifest** and a **station–granule match table** before any jobs are submitted.
6. Optionally submits **one NISAR GCOV HDF5 granule per MAAP DPS job** using your registered algorithm `nisar-gcov-ease2-v1:main`, requesting **EASE-Grid 2.0 M0p05 (50 m)**, **sigma0 in linear power**, **Enhanced Lee 7×7**, **frequencyA**, and **no AOI cropping** (`clip_to_aoi=no`). Both ascending/descending passes are retained. LIA is off by default; change to `yes` if the MAAP DEM access is configured.
7. Maintains CMR query caches and a persistent job ledger so rerunning the notebook does not resubmit accepted jobs.

**Important:** `clip_to_aoi=no` means the **full original GCOV scene** is processed. The geographic search-bin and station coordinates do **not** crop GeoTIFF outputs. `50 m` means the exact EASE2 M0p05 grid spacing of approximately `50.044751167478 m` (EPSG:6933); sigma0 uses `rtcGammaToSigmaFactor` from the NISAR GCOV product.

**MAAP registration prerequisite:** The algorithm must have been registered with a required **File Input** `input_file`, followed by these **Positional Inputs in exactly this order**: `ease2_res`, `mode`, `filter_window`, `lia`, `frequency`, `backscatter`, `clip_to_aoi`. Registering only `input_file` is **not sufficient** to request 50 m sigma0. The runner receives positional values and expects this order. The optional `bbox_*` fields are not needed because we never clip. Use the registration guide in [the standalone repository](https://github.com/preet0lal/nisar-gcov-ease2-maap/blob/main/MAAP_REGISTRATION.md).

## Instructions

- Upload this notebook and `ismn_gt50_coords_points_june.csv` into **the same folder** of your MAAP Jupyter workspace.
- Run cells in order. First inspect the manifest with `SUBMIT_JOBS=False`.
- Confirm MAAP registration input order; set `CONFIRM_MAAP_INPUT_ORDER=True`, `SUBMIT_JOBS=True`, and `MAX_NEW_SUBMISSIONS=1` for a smoke test.
- Check that the output covers the **whole source scene** and filenames contain `EASE2_50m_sigma0_enhanced_lee_w7`. Afterwards use `MAX_NEW_SUBMISSIONS=None` to submit the remaining manifest. Start with a conservative `MAX_IN_FLIGHT` because 50 m scenes are large.
- Do **not** run two simultaneous copies of the controller for the same `BATCH_IDENTIFIER`.

No MAAP job is submitted until you explicitly turn on submission. The notebook itself performs catalog discovery, not local HDF5 downloads or SAR conversion.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone, date, timedelta
from urllib.parse import urlparse
import hashlib, json, math, re, time, warnings

import pandas as pd
import earthaccess
from shapely.geometry import Point, Polygon, box
from shapely.ops import unary_union
from maap.maap import MAAP
from IPython.display import display

maap = MAAP()

# ---- INPUTS AND DATES ----
CSV_PATH = Path('ismn_gt50_coords_points_june.csv')
START_DATE = '2026-06-01'                    # ALL acquisitions on/after June 1, 2026
END_DATE = datetime.now(timezone.utc).date().isoformat()  # dynamically current UTC date
COLLECTIONS = {
    'provisional': 'NISAR_L2_GCOV_PROVISIONAL_V1',
    'validated': 'NISAR_L2_GCOV_V1',
}
PASSES = {'A', 'D'}
FREQUENCY = 'frequencyA'                    # original NISAR L-band frequency A
POLARIZATION_FILTER = None                  # set to a filename polarization code to restrict

# CMR spatial/time queries. BINS ARE FOR DISCOVERY ONLY, NEVER FOR CROPPING.
SEARCH_BIN_DEG = 5.0                       # bins with ≥1 station, not a continent-wide bbox
SEARCH_DAYS = 60                          # date slices to avoid huge CMR responses
USE_CMR_CACHE = True
REFRESH_CMR_CACHE = False

# ---- YOUR REGISTERED MAAP DPS ALGORITHM ----
ALGORITHM = 'nisar-gcov-ease2-v1'
VERSION = 'main'
WORKER_QUEUE = 'maap-dps-worker-32vcpu-64gb'  # choose an available MAAP queue
BATCH_IDENTIFIER = 'ISMN_NISAR_GCOV_JUNE2026_EASE2_50M_SIGMA0_EL7'

EASE2_RES = '50'
BACKSCATTER = 'sigma0'
MODE = 'enhanced_lee'                       # retain 7x7 filter from supplied Indonesia notebook
FILTER_WINDOW = '7'
LIA = 'no'                                 # change to yes after DEM credentials test
CLIP_TO_AOI = 'no'                         # MUST remain 'no' to keep complete GCOV scenes

# ---- SAFE SUBMISSION SETTINGS ----
SUBMIT_JOBS = False                        # inspect manifest before setting True
CONFIRM_MAAP_INPUT_ORDER = False           # True ONLY after verifying registration positional order
MAX_NEW_SUBMISSIONS = 1                    # first one granule; None submits all not-yet-known
MAX_IN_FLIGHT = 24                         # conservative starting cap for 50 m processing
POLL_SECONDS = 90

PRIVATE_CANDIDATES = [Path('/my-private-bucket'), Path.home()/'my-private-bucket',
                      Path('/projects/my-private-bucket')]
PRIVATE = next((p for p in PRIVATE_CANDIDATES if p.exists()), None)
if PRIVATE is None:
    raise RuntimeError('MAAP private bucket not mounted. Run in MAAP ADE, or modify PRIVATE_CANDIDATES.')
OUT = PRIVATE / 'ISMN_NISAR_GCOV_EASE2_50M_SIGMA0' / BATCH_IDENTIFIER
CACHE_DIR = OUT / 'cmr_cache'
OUT.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)
MANIFEST_CSV = OUT / 'granule_manifest.csv'
MATCHES_CSV = OUT / 'station_granule_matches.csv'
JOBS_CSV = OUT / 'maap_jobs.csv'

assert CSV_PATH.is_file(), f'Upload station CSV alongside the notebook: {CSV_PATH.resolve()}'
assert date.fromisoformat(START_DATE) <= date.fromisoformat(END_DATE)
assert CLIP_TO_AOI == 'no', 'This notebook is for FULL frames only'
assert EASE2_RES == '50' and BACKSCATTER == 'sigma0'
assert MAX_IN_FLIGHT >= 1 and SEARCH_DAYS >= 1 and SEARCH_BIN_DEG > 0
print('Date window:', START_DATE, '→', END_DATE, '(UTC; both dates included)')
print('Collections:', list(COLLECTIONS.values()))
print('Algorithm:', ALGORITHM, ':', VERSION)
print('Processing:', EASE2_RES, 'm EASE2,', BACKSCATTER, MODE, FILTER_WINDOW+'×'+FILTER_WINDOW,
      'LIA='+LIA, 'clip='+CLIP_TO_AOI)
print('Saved manifests and ledger:', OUT)
print('SUBMIT_JOBS:', SUBMIT_JOBS, '| MAX_NEW_SUBMISSIONS:', MAX_NEW_SUBMISSIONS)


In [ ]:
# Read and validate the EXACT coordinates from the provided CSV.
stations = pd.read_csv(CSV_PATH, dtype={'network': str, 'station': str})
required = ['network', 'station', 'lat', 'lon', 'n_points_june_onwards']
missing = sorted(set(required) - set(stations.columns))
if missing:
    raise ValueError(f'Missing station CSV columns: {missing}')
for key in ('lat', 'lon', 'n_points_june_onwards'):
    stations[key] = pd.to_numeric(stations[key], errors='raise')
if stations[required].isna().any().any():
    raise ValueError('Station CSV has missing required values')
if not stations.lat.between(-90, 90).all() or not stations.lon.between(-180, 180).all():
    raise ValueError('Invalid station lat/lon')
if stations.duplicated(['network', 'station']).any():
    raise ValueError('Duplicate network/station names; need unique station identifiers')

stations['station_id'] = stations.network.astype(str) + '::' + stations.station.astype(str)
source_sha256 = hashlib.sha256(CSV_PATH.read_bytes()).hexdigest()

# All geographic bins containing at least one station. Searching entire bins
# captures ANY granule containing a station, then a more precise point-in-footprint
# check removes irrelevant frames. This does not clip any SAR pixels.
def station_bin(lat, lon):
    return (int(math.floor(lon / SEARCH_BIN_DEG)), int(math.floor(lat / SEARCH_BIN_DEG)))

bin_to_sites = defaultdict(list)
site_by_id = {}
for rec in stations.to_dict('records'):
    station = {k: rec[k] for k in required}
    station['station_id'] = rec['station_id']
    bin_to_sites[station_bin(rec['lat'], rec['lon'])].append(station)
    site_by_id[rec['station_id']] = station

SEARCH_BINS = []
for (ix, iy), sites in sorted(bin_to_sites.items()):
    west, south = ix*SEARCH_BIN_DEG, iy*SEARCH_BIN_DEG
    bbox = (max(-180., west), max(-90., south), min(180., west+SEARCH_BIN_DEG),
            min(90., south+SEARCH_BIN_DEG))
    SEARCH_BINS.append((ix, iy, bbox, sites))

print(f'Stations: {len(stations)} | distinct search bins: {len(SEARCH_BINS)}')
print('Networks:', stations.network.value_counts().to_dict())
print('Station geographic range: lon', (stations.lon.min(), stations.lon.max()),
      'lat', (stations.lat.min(), stations.lat.max()))
display(stations.head(10))


In [ ]:
# CMR metadata helpers: filename parsing, physical geometry, HDF5 URL selection.
# Collection name (NOT filename prefix) determines provisional vs validated.
def granule_name(g):
    meta = g.get('meta', {})
    umm = g.get('umm', {})
    candidates = [meta.get('native-id'), meta.get('producer-granule-id'),
                  meta.get('title'), umm.get('GranuleUR')]
    for raw in candidates:
        if not raw:
            continue
        name = str(raw).split('?')[0].rstrip('/').split('/')[-1]
        name = re.sub(r'\.(h5|hdf5)$', '', name, flags=re.I)
        if name.startswith('NISAR_') and '_GCOV_' in name:
            return name
    return None


def parse_gcov_name(name):
    # NISAR_L2_PR_GCOV_cycle_track_pass_frame_mode_pol_source_start_end_CRID_*_*_*_counter
    parts = name.split('_')
    if len(parts) != 18 or parts[0] != 'NISAR' or parts[1] != 'L2' or parts[3] != 'GCOV':
        return None
    try:
        assert parts[6] in ('A', 'D')
        assert re.fullmatch(r'\d{8}T\d{6}', parts[11])
        assert re.fullmatch(r'\d{8}T\d{6}', parts[12])
        assert re.fullmatch(r'[A-Z]\d{5}', parts[13], re.I)
        return dict(cycle=int(parts[4]), track=int(parts[5]), pass_dir=parts[6],
                    frame=int(parts[7]), bw_mode=parts[8], polarization=parts[9],
                    acquisition_source=parts[10], date=parts[11][:8],
                    start_time=parts[11], end_time=parts[12], crid=parts[13],
                    crid_number=int(parts[13][1:]), coverage=parts[15],
                    counter=int(parts[17]))
    except (ValueError, AssertionError, IndexError):
        return None


def cmr_geometry(g):
    # Prefer actual granule polygons, then CMR bounding rectangles/boxes.
    umm = g.get('umm', {})
    geom = (umm.get('SpatialExtent', {}).get('HorizontalSpatialDomain', {})
            .get('Geometry', {}))
    actual = []
    for item in geom.get('GPolygons', []) or []:
        pts = item.get('Boundary', {}).get('Points', []) or []
        xy = [(float(p['Longitude']), float(p['Latitude'])) for p in pts
              if 'Longitude' in p and 'Latitude' in p]
        if len(xy) >= 3:
            polygon = Polygon(xy)
            if not polygon.is_valid:
                polygon = polygon.buffer(0)
            if not polygon.is_empty:
                actual.append(polygon)
    if actual:
        return unary_union(actual), 'polygon'
    rects = []
    for b in geom.get('BoundingRectangles', []) or []:
        try:
            rects.append(box(float(b['WestBoundingCoordinate']), float(b['SouthBoundingCoordinate']),
                             float(b['EastBoundingCoordinate']), float(b['NorthBoundingCoordinate'])))
        except (KeyError, TypeError, ValueError):
            continue
    if rects:
        return unary_union(rects), 'bounding_rectangle'
    for bounds in g.get('meta', {}).get('boxes', []) or []:
        try:
            s, w, n, e = map(float, bounds.split())
            return box(w, s, e, n), 'cmr_box_fallback'
        except (AttributeError, ValueError):
            continue
    return None, 'missing'


def gcov_hdf5_url(g, name):
    urls = []
    for access in ('direct', 'external'):
        try:
            urls.extend(g.data_links(access=access))
        except (TypeError, ValueError, AttributeError):
            pass
    for item in g.get('umm', {}).get('RelatedUrls', []) or []:
        if isinstance(item, dict) and item.get('URL'):
            urls.append(item['URL'])
    candidates = []
    for uri in dict.fromkeys(map(str, urls)):
        path = urlparse(uri).path
        filename = path.rstrip('/').split('/')[-1]
        if filename.lower() not in (name.lower()+'.h5', name.lower()+'.hdf5'):
            continue  # exclude *_QA_STATS.h5, quicklooks, ancillary files
        if uri.startswith('s3://'):
            priority = 0
        elif uri.startswith('https://'):
            priority = 1
        else:
            priority = 2
        candidates.append((priority, uri))
    candidates.sort()
    return candidates[0][1] if candidates else None


def dedup_identity(item):
    # Same physical GCOV acquisition across processor releases/CRIDs.
    return tuple(item[key] for key in ('cycle','track','pass_dir','frame',
                 'bw_mode','polarization','acquisition_source','start_time','end_time'))

print('CMR helpers loaded; full-frame processing: no bbox passed to MAAP.')


In [ ]:
# Discover provisional + validated GCOV over station-containing bins and 60-day
# time slices. All queries are cached (per date, collection, bin & CSV hash).
# Re-running this cell is safe; no MAAP jobs are submitted here.
start = date.fromisoformat(START_DATE)
end_exclusive = date.fromisoformat(END_DATE) + timedelta(days=1)
all_granules = {}               # exact filename -> full granule metadata
station_hits = defaultdict(dict)  # filename -> station_id -> match record
query_stats = Counter()

for maturity, collection in COLLECTIONS.items():
    t0 = start
    while t0 < end_exclusive:
        t1 = min(t0 + timedelta(days=SEARCH_DAYS), end_exclusive)
        temporal = (f'{t0.isoformat()}T00:00:00Z', f'{t1.isoformat()}T00:00:00Z')
        for ix, iy, search_bbox, sites in SEARCH_BINS:
            params = dict(collection=collection, temporal=list(temporal), bbox=list(search_bbox),
                          stations_sha256=source_sha256, bin_size=SEARCH_BIN_DEG,
                          matching_version=3, polarization_filter=POLARIZATION_FILTER)
            sig = hashlib.sha256(json.dumps(params, sort_keys=True).encode()).hexdigest()[:20]
            cache_path = CACHE_DIR / f'{maturity}_{t0:%Y%m%d}_bin{ix}_{iy}_{sig}.json'
            if USE_CMR_CACHE and not REFRESH_CMR_CACHE and cache_path.exists():
                cache = json.loads(cache_path.read_text())
                records = cache['records'] if cache.get('params') == params else None
            else:
                records = None
            if records is None:
                # Earthaccess/CMR searches the *intersecting full granules*;
                # site-in-footprint filtering happens below, not in processing.
                results = earthaccess.search_data(short_name=collection,
                    temporal=temporal, bounding_box=search_bbox, count=-1)
                query_stats['live_queries'] += 1
                records = []
                for g in results:
                    name = granule_name(g)
                    meta = parse_gcov_name(name) if name else None
                    if meta is None:
                        query_stats['unparsed_granules'] += 1
                        continue
                    if meta['pass_dir'] not in PASSES:
                        continue
                    if POLARIZATION_FILTER is not None and meta['polarization'] != POLARIZATION_FILTER:
                        continue
                    # Enforce acquisition date from filename in addition to CMR temporal search.
                    acq = date.fromisoformat(meta['date'][:4]+'-'+meta['date'][4:6]+'-'+meta['date'][6:8])
                    if not (start <= acq < end_exclusive):
                        continue
                    footprint, geometry_type = cmr_geometry(g)
                    if footprint is None:
                        query_stats['missing_cmr_footprints'] += 1
                        continue  # never guess that a broad search-bin hit covers a station
                    matches = [dict(station_id=site['station_id'],
                                    geometry_type=geometry_type,
                                    footprint_match=(geometry_type=='polygon'))
                               for site in sites
                               if footprint.covers(Point(site['lon'], site['lat']))]
                    if not matches:
                        continue
                    uri = gcov_hdf5_url(g, name)
                    if not uri:
                        query_stats['missing_main_gcov_hdf5_link'] += 1
                        continue
                    records.append(dict(granule=name, input_file=uri, collection=collection,
                                        maturity=maturity, matches=matches, **meta))
                if USE_CMR_CACHE:
                    # Save successful searches only; a failing CMR call is not cached.
                    cache_path.write_text(json.dumps({'params': params, 'records': records},
                                                      separators=(',', ':')))
            else:
                query_stats['cache_hits'] += 1
            for record in records:
                name = record['granule']
                candidate = {key: value for key, value in record.items() if key != 'matches'}
                if name not in all_granules:
                    all_granules[name] = candidate
                for site in record['matches']:
                    station_hits[name][site['station_id']] = site
                query_stats['station_frame_hits'] += len(record['matches'])
        print(f'{maturity:11s} {t0} — {t1-timedelta(days=1)} : '
              f'{len(all_granules):,} distinct granules | '
              f'{query_stats["live_queries"]} live queries, '
              f'{query_stats["cache_hits"]} cached')
        t0 = t1

print('CMR discovery finished:', dict(query_stats))
print('Unique GCOV filenames matching ≥1 station:', len(all_granules))
if not all_granules:
    warnings.warn('No matching granules. Inspect CMR access, date interval, collection names and station locations.')
if query_stats['missing_cmr_footprints']:
    warnings.warn(f'{query_stats["missing_cmr_footprints"]} hits had missing CMR geometry and were excluded '
                  'rather than mistakenly assigning a frame to an unrelated station.')


In [ ]:
# Build ONE row per distinct GCOV HDF5 scene and ONE row per station-scene pair.
# For the same physical acquisition: validated > provisional, then highest CRID/counter.
raw = list(all_granules.values())
variants_by_identity = defaultdict(list)
for variant in raw:
    variants_by_identity[dedup_identity(variant)].append(variant)
selected_by_acq = {}
for item in raw:
    key = dedup_identity(item)
    score = (1 if item['maturity'] == 'validated' else 0,
             int(item['crid_number']), int(item['counter']))
    old = selected_by_acq.get(key)
    if old is None or score > old[0]:
        selected_by_acq[key] = (score, item)

selected = [pair[1] for pair in selected_by_acq.values()]
selected.sort(key=lambda r: (r['date'], r['start_time'], r['track'], r['frame']))
scene_records = []
match_records = []
for item in selected:
    # Union station assignments from superseded versions of this SAME acquisition.
    identity = dedup_identity(item)
    all_sites_for_scene = {}
    for variant in variants_by_identity[identity]:
        all_sites_for_scene.update(station_hits[variant['granule']])
    for sid, match in sorted(all_sites_for_scene.items()):
        st = site_by_id[sid]
        match_records.append(dict(granule=item['granule'], maturity=item['maturity'],
            collection=item['collection'], date=item['date'], pass_dir=item['pass_dir'],
            track=item['track'], frame=item['frame'], station_id=sid,
            network=st['network'], station=st['station'], lat=st['lat'], lon=st['lon'],
            n_points_june_onwards=st['n_points_june_onwards'],
            geometry_type=match['geometry_type']))
    scene_records.append({**item, 'matched_stations':len(all_sites_for_scene),
                          'station_ids':' | '.join(sorted(all_sites_for_scene))})

manifest = pd.DataFrame(scene_records)
matches = pd.DataFrame(match_records)
manifest_columns = ['granule','maturity','collection','date','pass_dir','track','frame',
    'cycle','bw_mode','polarization','crid','input_file','matched_stations','station_ids']
match_columns = ['granule','maturity','date','pass_dir','track','frame',
    'station_id','network','station','lat','lon','n_points_june_onwards','geometry_type']
manifest = manifest.reindex(columns=manifest_columns)
matches = matches.reindex(columns=match_columns)
manifest.to_csv(MANIFEST_CSV, index=False)
matches.to_csv(MATCHES_CSV, index=False)
print('Station count:', len(stations), '| selected FULL GCOV frames:', len(manifest),
      '| station–frame matches:', len(matches),
      '| unique stations covered:', matches.station_id.nunique())
if len(manifest):
    print('Frames by maturity:', manifest.maturity.value_counts().to_dict())
    print('Frames by pass:', manifest.pass_dir.value_counts().to_dict())
    print('Frames by acquisition month:',manifest.date.str.slice(0,6).value_counts().sort_index().to_dict())
    display(manifest[['date','maturity','pass_dir','track','frame','matched_stations','granule']].head(25))
if len(matches):
    covered = set(matches.station_id)
    not_covered = stations.loc[~stations.station_id.isin(covered),['network','station','lat','lon']]
    print('Stations with no CMR match:',len(not_covered))
    if len(not_covered):
        display(not_covered.head(20))
print('Granule manifest:', MANIFEST_CSV)
print('Station coverage table:', MATCHES_CSV)


In [ ]:
# Confirm algorithm exists; then VERIFY the MAAP registration input schema.
# The GitHub run_maap.sh accepts: input_file, ease2_res, mode, filter_window,
# lia, frequency, backscatter, clip_to_aoi (and optional bbox_* thereafter).
response = maap.listAlgorithms()
response.raise_for_status() if hasattr(response, 'raise_for_status') else None
catalog = response.json().get('algorithms', [])
registered = [a for a in catalog if a.get('type') == ALGORITHM and a.get('version') == VERSION]
print('Registered algorithm found:', bool(registered), f'{ALGORITHM}:{VERSION}')
if not registered:
    raise RuntimeError('Check Algorithm Name and Version in MAAP Submit Jobs. '
                       f'Could not find {ALGORITHM}:{VERSION}.')
print('MAAP catalog record (inspect available input metadata):')
print(json.dumps(registered[0], indent=2, default=str)[:6000])

expected_input_order = ['input_file','ease2_res','mode','filter_window','lia',
                        'frequency','backscatter','clip_to_aoi']
print('\nREQUIRED registered inputs, in order:\n  ' + ' → '.join(expected_input_order))
print('The remaining bbox_w, bbox_s, bbox_e, bbox_n inputs are OPTIONAL '
      'and NOT submitted because clip_to_aoi=no.')
print('Registration verified by user:', CONFIRM_MAAP_INPUT_ORDER)
if SUBMIT_JOBS and not CONFIRM_MAAP_INPUT_ORDER:
    raise RuntimeError('Before submitting, check your MAAP registration has the '
       'File Input + ALL SEVEN Positional Inputs in exact order shown above, '
       'then set CONFIRM_MAAP_INPUT_ORDER=True in settings. Otherwise 50m sigma0 '
       'might silently use 100m gamma0 defaults, or argument positions can shift.')


In [ ]:
# Restartable DPS job ledger and submission utilities.
# One job per unique granule; prevent duplicate submissions on rerun.
COLUMNS = ['granule','maturity','date','track','frame','pass_dir','job_id','status',
           'identifier','submitted_utc','checked_utc','error']
TERMINAL_SUCCESS = {'success','succeeded','completed','complete','job-completed'}
TERMINAL_FAILURE = {'failed','dismissed','cancelled','canceled','job-revoked',
                    'job-failed','job-canceled','error'}

def _ledger_read(path):
    if not path.exists():
        return pd.DataFrame(columns=COLUMNS, dtype=str)
    frame = pd.read_csv(path,dtype=str).fillna('')
    for field in COLUMNS:
        if field not in frame:
            frame[field] = ''
    return frame[COLUMNS]


def load_jobs():
    old = []
    for path in (JOBS_CSV, JOBS_CSV.with_suffix('.csv.tmp')):
        try:
            old.append(_ledger_read(path))
        except Exception as ex:
            print('WARNING: job ledger read failed:',path,repr(ex))
    combined = pd.concat(old,ignore_index=True) if old else pd.DataFrame(columns=COLUMNS)
    return combined.drop_duplicates('granule',keep='last').reset_index(drop=True)


def persist_jobs(frame):
    current = load_jobs()
    combined = pd.concat([current,frame],ignore_index=True).drop_duplicates('granule',keep='last')
    # Work around MAAP private bucket object mount: save both files, no os.replace.
    combined.to_csv(JOBS_CSV.with_suffix('.csv.tmp'),index=False)
    combined.to_csv(JOBS_CSV,index=False)


def maap_status(job_id):
    response = maap.getJobStatus(str(job_id))
    raw = response.text if hasattr(response,'text') else str(response)
    mt = re.search(r'<(?:\w+:)?Status>\s*([^<]+)',raw,re.I)
    return (mt.group(1) if mt else raw).strip().lower()


def poll_existing(jobs):
    dirty = 0
    for ix, rec in jobs.iterrows():
        if str(rec.status).lower() in TERMINAL_SUCCESS | TERMINAL_FAILURE:
            continue
        if not str(rec.job_id).strip():
            print('WARNING: submitted record lacks job ID:',rec.granule)
            continue
        try:
            jobs.at[ix,'status'] = maap_status(rec.job_id)
            jobs.at[ix,'checked_utc'] = datetime.now(timezone.utc).isoformat()
            dirty += 1
        except Exception as ex:
            print('MAAP job status unavailable:',rec.job_id,str(ex)[:150])
        if dirty and dirty % 25 == 0:
            persist_jobs(jobs)
    if dirty:
        persist_jobs(jobs)
    return jobs


def overview(jobs, total):
    status = jobs.status.str.lower() if len(jobs) else pd.Series([],dtype=str)
    success = int(status.isin(TERMINAL_SUCCESS).sum())
    failure = int(status.isin(TERMINAL_FAILURE).sum())
    inflight = len(jobs)-success-failure
    print(f'Status: successes={success}, failures={failure}, in-flight={inflight}, '
          f'not yet submitted={total-len(jobs)} | cap={MAX_IN_FLIGHT}')
    return inflight


def submit_kwargs(row):
    assert CLIP_TO_AOI=='no', 'Never crop full scenes in this controller'
    return dict(identifier=BATCH_IDENTIFIER, algo_id=ALGORITHM, version=VERSION,
        queue=WORKER_QUEUE, input_file=str(row.input_file),
        ease2_res=EASE2_RES, mode=MODE, filter_window=FILTER_WINDOW,
        lia=LIA, frequency=FREQUENCY, backscatter=BACKSCATTER, clip_to_aoi='no')

print('Submission example (no secret tokens):')
if len(manifest):
    print(json.dumps(submit_kwargs(next(manifest.itertuples(index=False))),indent=2))
print('Existing jobs recorded:',len(load_jobs()),'| ledger:',JOBS_CSV)


In [ ]:
# Run the controller. First run with SUBMIT_JOBS=False and inspect manifests.
# A successful first job should be inspected before setting MAX_NEW_SUBMISSIONS=None.
if manifest.empty:
    print('No matching NISAR granules; nothing to submit.')
elif not SUBMIT_JOBS:
    print('DRY RUN: no MAAP jobs submitted.')
    print('Full frames:', len(manifest), '| saved job records:',len(load_jobs()))
    print('To launch: verify positional registration; set CONFIRM_MAAP_INPUT_ORDER=True, '
          'SUBMIT_JOBS=True, MAX_NEW_SUBMISSIONS=1 and rerun the preflight + controller cells.')
else:
    if not CONFIRM_MAAP_INPUT_ORDER:
        raise RuntimeError('Must confirm your exact registered MAAP input order first.')
    sent = 0
    while True:
        jobs = poll_existing(load_jobs())
        pending = manifest[~manifest.granule.isin(set(jobs.granule))]
        in_flight = overview(jobs,len(manifest))
        if pending.empty or (MAX_NEW_SUBMISSIONS is not None and sent>=MAX_NEW_SUBMISSIONS):
            print('Nothing left to submit in this run.')
            break
        slots=max(0,MAX_IN_FLIGHT-in_flight)
        if MAX_NEW_SUBMISSIONS is not None:
            slots=min(slots,MAX_NEW_SUBMISSIONS-sent)
        for row in pending.head(slots).itertuples(index=False):
            params=submit_kwargs(row)
            try:
                result=maap.submitJob(**params)
                job_id=getattr(result,'id',None)
                if not job_id:
                    raise RuntimeError(f'MAAP did not return a job ID for {row.granule}: {result}')
                now=datetime.now(timezone.utc).isoformat()
                record=dict(granule=row.granule, maturity=row.maturity, date=row.date,
                    track=str(row.track), frame=str(row.frame), pass_dir=row.pass_dir,
                    job_id=str(job_id), status='accepted', identifier=BATCH_IDENTIFIER,
                    submitted_utc=now, checked_utc=now, error='')
                jobs=pd.concat([jobs,pd.DataFrame([record])],ignore_index=True)
                persist_jobs(jobs)  # Save each successful acceptance immediately
                sent+=1
                print(f'SUBMIT {sent}: {row.date} {row.pass_dir} '
                      f'track {row.track}, frame {row.frame}, {row.maturity} -> {job_id}')
                time.sleep(0.3)
            except Exception as exc:
                print('SUBMISSION ERROR:',row.granule,':',str(exc)[:400])
                print('Stopped to avoid possibly duplicating a job after an ambiguous response.')
                raise
        if MAX_NEW_SUBMISSIONS is not None and sent>=MAX_NEW_SUBMISSIONS:
            print('Reached MAX_NEW_SUBMISSIONS; inspect the job output before bulk submission.')
            break
        if len(pending)<=slots:
            print('All manifest entries are now submitted.')
            break
        print(f'Waiting {POLL_SECONDS}s for free MAAP slots...')
        time.sleep(POLL_SECONDS)


In [ ]:
# Run separately to refresh the saved job statuses whenever needed.
ledger=poll_existing(load_jobs())
overview(ledger,len(manifest))
if len(ledger):
    display(ledger[['date','maturity','pass_dir','track','frame','job_id','status']].tail(25))
print('MAAP job ledger:',JOBS_CSV)
print('CMR granule manifest:',MANIFEST_CSV)
print('Station-to-frame coverage:',MATCHES_CSV)


## Scientific and processing notes

- **Time:** acquisition UTC date from the NISAR granule filename must fall on/after `2026-06-01`; the end date is the date on which you run the notebook. ASF indicates provisional acquisitions begin in June 2026 and validated acquisitions begin in October 2026. The search deliberately includes both collections rather than assuming which has a given scene.
- **Product maturity:** only official provisional and validated **collection short names** are searched. BETA and UR are excluded. When identical acquisitions exist in both, the validated one is preferred; higher CRID/counter breaks ties within each maturity.
- **Geometry:** CMR polygons are checked against each coordinate. If CMR only provides a bounding rectangle, the hit is marked as an approximate `geometry_type` in `station_granule_matches.csv`. Missing geometry is excluded and counted (no unsupported claim that a scene intersects a station).
- **Radiometry:** the independent MAAP processor converts native linear gamma0 to sigma0 using `rtcGammaToSigmaFactor` **before** filtering and resampling; output GeoTIFFs retain **linear** backscatter, not dB. Quicklook images use dB for display only.
- **Masking:** CRID-sensitive exception masks are performed by the algorithm, not this notebook: provisional and validated use different stored layouts/validity tests.
- **Full scenes:** do not pass the station's search bin as an AOI crop. `clip_to_aoi=no` in every submission is intentional. A separate job runs for each distinct HDF5 granule; adjacent-frame mosaicking cannot occur across independent jobs.
- **Space and time:** full 50 m GCOV scenes may require substantial worker disk/RAM. Queue `maap-dps-worker-32vcpu-64gb` and modest in-flight cap are provisional choices, **not measured requirements**. Start with one smoke-test granule.
- **Config changes:** jobs already in the ledger are never automatically resubmitted, even if previously failed. For a different resolution/filter/date strategy, choose a new `BATCH_IDENTIFIER` so the manifest and ledger are not mixed.
- **Quality check:** open the first job's `output/` GeoTIFF with Rasterio; confirm CRS `EPSG:6933`, pixel size ≈`50.044751167478 m`, metadata `normalization=sigma0`, and bounds extending beyond the point-search bin where the source frame does. Confirm `*_EASE2_grid.mat` contains global pixel indices. Also check DPS log for `clip_to_aoi=no` and `filter=enhanced_lee`.

References: [ASF NISAR data search](https://nisar-docs.asf.alaska.edu/earthdata-search/) · [ASF NISAR release availability](https://nisar-docs.asf.alaska.edu/availability-overview/) · [your EASE2 MAAP repository](https://github.com/preet0lal/nisar-gcov-ease2-maap).
